In [0]:
Old_data = [(101, 'Rahul', 'Pune'), (102, 'Prachi', 'Mumbai')]
cols = ('id', 'name', 'city')
old_df = spark.createDataFrame(Old_data, cols)
display(old_df)

In [0]:
new_data = [(101, 'Rahul', 'Pune'), (102, 'Prachi', 'Hyderabad'), (103, 'Prachi', 'Mumbai')]
cols = ('id', 'name', 'city')
new_df = spark.createDataFrame(new_data, cols)
display(new_df)

# SCD Type 1 : Overwrite (No History)

In [0]:
#save the old data as delta format

from delta.tables import *
old_df.write.format("delta").mode("overwrite").saveAsTable("old_customer_records")
dim_customer = DeltaTable.forName(spark, "old_customer_records")

In [0]:
# Merge  for SCD type 1 
dim_customer.alias("target").merge(
    new_df.alias("source"),
    "target.id = source.id") \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

# SCD Type 2 : Keep History 

In [0]:
# Add audit fields to new

from pyspark.sql.functions import lit, current_date

df_new_sc2 = new_df.withColumn('is_active', lit(True)) \
.withColumn('start_date', current_date()) \
.withColumn('end_date', lit(None).cast('date'))



In [0]:
# add Audit filed to old data

df_old_sc2 = old_df.withColumn('is_active', lit(True)) \
.withColumn('start_date', lit('2024-01-01').cast('date')) \
.withColumn('end_date', lit(None).cast('date'))


In [0]:
# add Audit filed to new data

df_new_sc2 = new_df.withColumn('is_active', lit(True)) \
.withColumn('start_date', lit('2024-01-01').cast('date')) \
.withColumn('end_date', lit(None).cast('date'))


In [0]:
display(df_new_sc2)

In [0]:
# Expired change record 

df_expired = df_old_sc2.alias('old').join(new_df.alias('new'), 'id') \
.filter('old.city != new.city') \
.select('old.id', 'old.name', 'old.city')  \
.withColumn('is_active', lit(False)) \
.withColumn('start_date', lit('2024-01-01').cast('date')) \
.withColumn('end_date', current_date())

display(df_expired)




In [0]:
df_unchanged = df_old_sc2.alias('old').join(new_df.alias('new'), 'id') \
.filter('old.city = new.city')
display(df_unchanged)

In [0]:
# NEW RECORD ENTRY 
df_new_only = new_df.alias('new').join(old_df.alias('old'), 'id', 'leftanti') \
.withColumn('is_active', lit(True)) \
.withColumn('start_date', current_date()) \
.withColumn('end_date', lit(None).cast('date'))

display(df_new_only)

# Databrick Sql Implementation 

In [0]:
%sql
CREATE OR REPLACE TABLE dim_customers(
    customer_id int,
    name string,
    city string
) USING DELTA;

INSERT INTO dim_customers VALUES (101, 'Rahul', 'Pune'),
(102, 'Prachi', 'Mumbai');


In [0]:
%sql

--step2: Create Staging view

CREATE OR REPLACE TEMP VIEW Stage_Customer AS 
SELECT * FROM VALUES 
(101, 'Rahul', 'Pune'),
(102, 'Prachi', 'Mumbai'),
(103, 'Rajesh', 'Delhi')
AS t(customer_id, name, city)


In [0]:
%sql
SELECT * FROM Stage_Customer

# SCD Type 1 

In [0]:
%sql
MERGE INTO dim_customers AS target
USING Stage_Customer AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
  UPDATE SET
    target.name = source.name,
    target.city = source.city
WHEN NOT MATCHED THEN
  INSERT (customer_id, name, city)
  VALUES (source.customer_id, source.name, source.city);

In [0]:
%sql
Select * from dim_customers

# SCD TYPE 2